###import libraries

In [1]:
pip install numpy pandas matplotlib seaborn 

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Note: you may need to restart the kernel to use updated packages.


###import modules

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


###load dataset

In [3]:
df = pd.read_csv('C:\\Users\\Admin\\Downloads\\Sales Data Cleaning Task.csv')

print("Dataset loaded successfully")
print("Shape:", df.shape)

Dataset loaded successfully
Shape: (120, 11)


### UNDERSTANDING THE DATA

In [4]:

print("\n--- First 5 Records ---")
print(df.head())

print("\n--- Dataset Shape ---")
print(df.shape)

print("\n--- Dataset Information ---")
print(df.info())

print("\n--- Statistical Description ---")
print(df.describe(include="all"))

print("\n--- Column Names ---")
print(df.columns.tolist())


--- First 5 Records ---
  Order_ID  Order_Date Region     Product     Category Quantity Unit_Price  \
0    S0001  2026-01-27  North  Smartphone  Electronics      1.0    28000.0   
1    S0002  2026-06-04  North      Laptop  Electronics      1.0    55000.0   
2    S0003  2026-01-02  South      Tablet  Electronics      4.0    22000.0   
3    S0004  2026-03-28  North       Mouse  Accessories      5.0      400.0   
4    S0005  2026-01-12   West      Tablet  Electronics      2.0    22000.0   

   Discount    Sales   Profit Salesperson  
0      0.05  26600.0  3137.36       Vijay  
1      0.05  52250.0  6246.61       Divya  
2      0.20  70400.0  8961.39       Vijay  
3      0.05   1900.0   461.18       Priya  
4      0.10  39600.0  7232.28       Priya  

--- Dataset Shape ---
(120, 11)

--- Dataset Information ---
<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0

### DATA CLEANING

In [5]:
clean = df.copy()

### Check Missing Values

In [6]:
print("\n--- Missing Values Before Cleaning ---")
print(clean.isnull().sum())


--- Missing Values Before Cleaning ---
Order_ID       0
Order_Date     1
Region         2
Product        1
Category       0
Quantity       1
Unit_Price     1
Discount       0
Sales          1
Profit         1
Salesperson    2
dtype: int64


### Check Empty Strings and Whitespace

In [7]:

print("\n--- Blank / Whitespace Values ---")

for column in clean.columns:
    blank_count = (
        clean[column]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )

    if blank_count > 0:
        print(column, ":", blank_count)


--- Blank / Whitespace Values ---
Product : 1


###  Clean Text Columns

In [8]:
text_columns = [
    "Order_ID",
    "Region",
    "Product",
    "Category",
    "Salesperson"
]

for column in text_columns:
    clean[column] = (
        clean[column]
        .astype("string")
        .str.strip()
    )

### Convert empty strings to missing values

In [9]:
clean = clean.replace(
    r"^\s*$",
    pd.NA,
    regex=True
)

### Standardize capitalization

In [10]:
for column in [
    "Region",
    "Product",
    "Category",
    "Salesperson"
]:
    clean[column] = clean[column].str.title()

### CORRECT DATA TYPES

# Order Date

In [11]:
clean["Order_Date"] = pd.to_datetime(
    clean["Order_Date"],
    errors="coerce",
    format="mixed"
)

# Quantity

# Correct text values found in the dataset

In [12]:
clean["Quantity"] = clean["Quantity"].replace({
    "five": "5",
    "3 units": "3"
})

clean["Quantity"] = pd.to_numeric(
    clean["Quantity"],
    errors="coerce"
)

# Unit Price

In [13]:
clean["Unit_Price"] = (
    clean["Unit_Price"]
    .astype("string")
    .str.replace(
        r"\s*INR\s*$",
        "",
        regex=True
    )
)

clean["Unit_Price"] = pd.to_numeric(
    clean["Unit_Price"],
    errors="coerce"
)

# Discount, Sales and Profit

In [14]:

for column in [
    "Discount",
    "Sales",
    "Profit"
]:
    clean[column] = pd.to_numeric(
        clean[column],
        errors="coerce"
    )

### DETECT INVALID VALUES

# Quantity must be greater than zero

In [15]:
invalid_quantity = (
    clean["Quantity"].notna()
    & (clean["Quantity"] <= 0)
)

# Unit price must be greater than zero

In [16]:
invalid_price = (
    clean["Unit_Price"].notna()
    & (clean["Unit_Price"] <= 0)
)

# Discount should be between 0 and 1

In [17]:
invalid_discount = (
    clean["Discount"].notna()
    & (
        (clean["Discount"] < 0)
        | (clean["Discount"] > 1)
    )
)

# Sales should not be negative

In [19]:
invalid_sales = (
    clean["Sales"].notna()
    & (clean["Sales"] < 0)
)


print("\n--- Invalid Quantity ---")
print(clean.loc[
    invalid_quantity,
    ["Order_ID", "Quantity"]
])

print("\n--- Invalid Unit Price ---")
print(clean.loc[
    invalid_price,
    ["Order_ID", "Unit_Price"]
])

print("\n--- Invalid Discount ---")
print(clean.loc[
    invalid_discount,
    ["Order_ID", "Discount"]
])

print("\n--- Invalid Sales ---")
print(clean.loc[
    invalid_sales,
    ["Order_ID", "Sales"]
])


--- Invalid Quantity ---
   Order_ID  Quantity
31    S0032      -2.0

--- Invalid Unit Price ---
   Order_ID  Unit_Price
54    S0055     -5000.0

--- Invalid Discount ---
   Order_ID  Discount
76    S0077      1.25

--- Invalid Sales ---
    Order_ID    Sales
110    S0111 -15000.0


### HANDLE INVALID VALUES

In [20]:
clean.loc[
    invalid_quantity,
    "Quantity"
] = np.nan

clean.loc[
    invalid_price,
    "Unit_Price"
] = np.nan

clean.loc[
    invalid_discount,
    "Discount"
] = np.nan

clean.loc[
    invalid_sales,
    "Sales"
] = np.nan

### HANDLE MISSING QUANTITY

# Business rule:
# Sales = Quantity × Unit_Price × (1 - Discount)

In [21]:
mask = (
    clean["Quantity"].isna()
    & ~invalid_quantity
    & clean["Sales"].notna()
    & clean["Unit_Price"].notna()
    & clean["Discount"].notna()
)

derived_quantity = (
    clean.loc[mask, "Sales"]
    /
    (
        clean.loc[mask, "Unit_Price"]
        *
        (1 - clean.loc[mask, "Discount"])
    )
)

for index, value in derived_quantity.items():

    if (
        np.isfinite(value)
        and value > 0
        and abs(value - round(value)) < 1e-9
    ):
        clean.loc[index, "Quantity"] = round(value)

### HANDLE MISSING UNIT PRICE

In [22]:
mask = (
    clean["Unit_Price"].isna()
    & ~invalid_price
    & clean["Sales"].notna()
    & clean["Quantity"].notna()
    & clean["Discount"].notna()
)

derived_price = (
    clean.loc[mask, "Sales"]
    /
    (
        clean.loc[mask, "Quantity"]
        *
        (1 - clean.loc[mask, "Discount"])
    )
)

for index, value in derived_price.items():

    if (
        np.isfinite(value)
        and value > 0
    ):
        clean.loc[
            index,
            "Unit_Price"
        ] = round(value, 2)

### CATEGORY CONSISTENCY
# All other Laptop records are Electronics.
# One Laptop was incorrectly labelled Wearables.

In [23]:
clean.loc[
    (
        clean["Product"] == "Laptop"
    )
    &
    (
        clean["Category"] == "Wearables"
    ),
    "Category"
] = "Electronics"


### BUSINESS RULE VALIDATION

In [24]:
expected_sales = (
    clean["Quantity"]
    *
    clean["Unit_Price"]
    *
    (1 - clean["Discount"])
)

valid_inputs = clean[
    [
        "Quantity",
        "Unit_Price",
        "Discount"
    ]
].notna().all(axis=1)

# Recalculate Sales only when all required inputs are valid

In [25]:
clean.loc[
    valid_inputs,
    "Sales"
] = (
    expected_sales[
        valid_inputs
    ].round(2)
)

# If required inputs are missing/invalid,
# do not invent a Sales value.

In [26]:
clean.loc[
    ~valid_inputs,
    "Sales"
] = np.nan

### HANDLE MISSING CATEGORICAL VALUES

In [27]:
for column in [
    "Region",
    "Product",
    "Salesperson"
]:

    clean[column] = clean[column].fillna(
        "Unknown"
    )

### REMOVE DUPLICATE RECORDS

In [28]:
print("\nExact duplicates before removal:",
      clean.duplicated().sum())

clean = clean.drop_duplicates()

print("Exact duplicates after removal:",
      clean.duplicated().sum())


Exact duplicates before removal: 2
Exact duplicates after removal: 0


### FINAL DATA TYPES

In [29]:
clean["Order_ID"] = clean["Order_ID"].astype("string")

clean["Order_Date"] = pd.to_datetime(
    clean["Order_Date"]
)

clean["Quantity"] = clean["Quantity"].astype("Int64")

clean["Unit_Price"] = clean["Unit_Price"].astype(
    "Float64"
)

clean["Discount"] = clean["Discount"].astype(
    "Float64"
)

clean["Sales"] = clean["Sales"].astype(
    "Float64"
)

clean["Profit"] = clean["Profit"].astype(
    "Float64"
)

### FINAL DATA QUALITY VALIDATION

In [30]:
print("\n====================================")
print("FINAL DATA QUALITY VALIDATION")
print("====================================")

print("\nFinal Shape:")
print(clean.shape)

print("\nMissing Values:")
print(clean.isnull().sum())

print("\nDuplicate Rows:")
print(clean.duplicated().sum())

print("\nDuplicate Order IDs:")
print(clean["Order_ID"].duplicated().sum())

print("\nData Types:")
print(clean.dtypes)



FINAL DATA QUALITY VALIDATION

Final Shape:
(118, 11)

Missing Values:
Order_ID       0
Order_Date     2
Region         0
Product        0
Category       0
Quantity       1
Unit_Price     1
Discount       1
Sales          3
Profit         1
Salesperson    0
dtype: int64

Duplicate Rows:
0

Duplicate Order IDs:
0

Data Types:
Order_ID               string
Order_Date     datetime64[us]
Region                 string
Product                string
Category               string
Quantity                Int64
Unit_Price            Float64
Discount              Float64
Sales                 Float64
Profit                Float64
Salesperson            string
dtype: object


# Check invalid values again

In [31]:
print("\nInvalid Quantity:")
print(
    (
        (clean["Quantity"] <= 0)
        .fillna(False)
    ).sum()
)

print("\nInvalid Unit Price:")
print(
    (
        (clean["Unit_Price"] <= 0)
        .fillna(False)
    ).sum()
)

print("\nInvalid Discount:")
print(
    (
        (
            (clean["Discount"] < 0)
            |
            (clean["Discount"] > 1)
        )
        .fillna(False)
    ).sum()
)

print("\nInvalid Sales:")
print(
    (
        (clean["Sales"] < 0)
        .fillna(False)
    ).sum()
)


Invalid Quantity:
0

Invalid Unit Price:
0

Invalid Discount:
0

Invalid Sales:
0


# Check Sales Formula

In [32]:
final_expected_sales = (
    clean["Quantity"].astype("Float64")
    *
    clean["Unit_Price"]
    *
    (1 - clean["Discount"])
)

valid_rows = clean[
    [
        "Quantity",
        "Unit_Price",
        "Discount",
        "Sales"
    ]
].notna().all(axis=1)

sales_mismatch = (
    (
        clean["Sales"]
        -
        final_expected_sales
    ).abs() > 0.01
) & valid_rows

print("\nSales Formula Mismatches:")
print(sales_mismatch.sum())


Sales Formula Mismatches:
0


### KPI CALCULATION

In [33]:
total_sales = clean["Sales"].sum()

total_profit = clean["Profit"].sum()

total_quantity = clean["Quantity"].sum()

average_order_value = (
    total_sales
    /
    clean["Order_ID"].nunique()
)


print("\n====================================")
print("KEY PERFORMANCE INDICATORS")
print("====================================")

print(
    "Total Sales:",
    round(total_sales, 2)
)

print(
    "Total Profit:",
    round(total_profit, 2)
)

print(
    "Total Quantity Sold:",
    total_quantity
)

print(
    "Average Order Value:",
    round(average_order_value, 2)
)


KEY PERFORMANCE INDICATORS
Total Sales: 8349252.5
Total Profit: 1484938.86
Total Quantity Sold: 523
Average Order Value: 70756.38


### CATEGORY CONSISTENCY

In [34]:
print("\nCategory Counts:")
print(
    clean["Category"].value_counts()
)


Category Counts:
Category
Electronics    71
Accessories    35
Wearables      12
Name: count, dtype: Int64


### EXPORT CLEANED DATASET

In [35]:
clean.to_csv(
    "Sales Data Cleaned.csv",
    index=False
)

print(
    "\nCleaned dataset saved as: "
    "Sales Data Cleaned.csv"
)


Cleaned dataset saved as: Sales Data Cleaned.csv


## Before Cleaning
Dataset contained 120 records with duplicate rows and Order_IDs.
Missing values were present in dates and numeric fields.
Quantity, Unit_Price, Discount, and Sales contained invalid values.
Mixed formats included “five”, “3 units”, and “18000 INR”.
Region, Product, and Category had inconsistent capitalization and labels.
Some Sales values did not match the business calculation.
## After Cleaning
Reduced to 118 valid records after removing duplicates.
Order_IDs are now unique.
Numeric fields were converted to appropriate data types.
Text values were standardized for consistency.
Invalid values were corrected or retained as missing when unsupported.
Sales were validated using the business rule, making the dataset analysis-ready.